In [1]:
import gc
import os
import glob
from PIL import Image
import numpy as np
import pandas as pd
import torchvision.io as io
from torchmetrics.classification import BinaryAveragePrecision, BinaryROC
from torch.utils.data import Dataset, DataLoader
import torch
from tqdm import tqdm
import yaml
import importlib
from pathlib import Path
import warnings
from huggingface_hub import hf_hub_download,login
login()
import yaml
from torch.nn import functional as F
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
%cd ../eomt

data_path_anomaly = "../Anomaly_Validation_Datasets/Validation_Dataset"

erfnet_path = "../trained_models/erfnet_pretrained.pth" # Pretrained ERFNet

coco_ft_config_path = "./configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml"
coco_ft_ckpt_path = "./checkpoints/eomt_finetuned_cityscapes_step5_best_exp3.pt"
cs_config_path="./configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"  # change to the config file
cs_ckpt_path="./checkpoints/eomt_cityscapes.bin"
coco_config_path= "./configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml" # Change the configuration path for coco
coco_ckpt_path="./checkpoints/eomt_coco.bin"
data_path = "../Cityscapes"

subsets = [
    "FS_LostFound_full",
    "fs_static",
    "RoadAnomaly",
    "RoadAnomaly21",
    "RoadObsticle21"
]
device=0

def load_model_and_data(config_path, ckpt_path, data_path, device):
    """Loads the dataset, encoder, network, and weights based on the config file."""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)

    # Load dataset module
    data_module_name, class_name = config["data"]["class_path"].rsplit(".", 1)
    data_module = getattr(importlib.import_module(data_module_name), class_name)
    data_module_kwargs = config["data"].get("init_args", {})

    batch_size = 4 # batched for faster evaluation
    num_workers = 2

    if "batch_size" in data_module_kwargs:
      batch_size = data_module_kwargs["batch_size"]
      data_module_kwargs.pop("batch_size")

    if "num_workers" in data_module_kwargs:
      num_workers = data_module_kwargs["num_workers"]
      data_module_kwargs.pop("num_workers")

    data = data_module(
        path=data_path,
        batch_size=batch_size,
        num_workers=num_workers,
        check_empty_targets=False,
        **data_module_kwargs
    ).setup()

    # Load encoder
    encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
    encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
    encoder = encoder_cls(img_size=data.img_size, **encoder_cfg.get("init_args", {}))

    # Load network
    network_cfg = config["model"]["init_args"]["network"]
    network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
    network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
    network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder" and k != "num_classes" and k != "masked_attn_enabled"}

    network = network_cls(
        masked_attn_enabled=False, # disabled for evaluation
        num_classes=data.num_classes,
        encoder=encoder,
        **network_kwargs,
    )

    # Load Lightning module
    lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
    model_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k != "network" and k != "num_classes" and k != "img_size"}

    if "stuff_classes" in config["data"].get("init_args", {}):
        model_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

    model = lit_cls(
        img_size=data.img_size,
        num_classes=data.num_classes,
        network=network,
        **model_kwargs,
    ).eval().to(device)

    # Inject LoRAs if present
    if "callbacks" in config["trainer"]:
        for cb_cfg in config["trainer"]["callbacks"]:
            if "InjectLoRACallback" in cb_cfg["class_path"]:
                # Dynamically import and instantiate the callback
                module_name, class_name = cb_cfg["class_path"].rsplit(".", 1)
                cb_cls = getattr(importlib.import_module(module_name), class_name)
                cb_args = cb_cfg.get("init_args", {})
                lora_cb = cb_cls(**cb_args)

                # Inject
                lora_cb.setup(None, model, "validate")
                model.to(device)

    # Load weights
    print(f"Loading weights from {ckpt_path}...")
    ckpt = torch.load(ckpt_path, map_location=f"cuda:{device}", weights_only=False)

    if "state_dict" in ckpt: # full checkpoint
        state_dict = ckpt["state_dict"]
    else: # only state dict checkpoint
        state_dict = ckpt

    model.load_state_dict(state_dict, strict=False)

    return model, data

c:\Users\user\OneDrive - Politecnico di Torino\Desktop\Appunti Poli\magistrale\Fondamentals\Project_fundamentals\eomt


In [2]:
def find_path(dir, name, possible_exts):
  base_name = os.path.splitext(name)[0]

  path = None
  for ext in possible_exts:
      candidate = os.path.join(dir, base_name + ext)
      if os.path.exists(candidate):
          path = candidate
          break

  if path is None:
      raise FileNotFoundError(f"No file found for {name}")

  return path

class AnomalyDataset(Dataset):
    def __init__(self, dataset_path, subset, input_transform=None, target_transform=None):
        self.img_dir = f"{dataset_path}/{subset}/images"
        self.mask_dir = f"{dataset_path}/{subset}/labels_masks"
        self.img_names = sorted(os.listdir(self.img_dir))
        self.possible_exts = [".png", ".jpg", ".jpeg", ".webp"]

        # Optional transforms, used by ERFNet
        self.input_transform = input_transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        name = self.img_names[idx]

        img_path = find_path(self.img_dir, name, self.possible_exts)
        mask_path = find_path(self.mask_dir, name, self.possible_exts)

        # Load image
        if self.input_transform is not None: # ERFNet
            # load as PIL for the transform
            img = Image.open(img_path).convert('RGB')
            img = self.input_transform(img)
        else: # EoMT
            img = io.read_image(img_path, mode=io.ImageReadMode.RGB)

        # Load GT mask
        mask_img = Image.open(mask_path)

        if self.target_transform is not None: # ERFNet
            mask_img = self.target_transform(mask_img)

        gt_bin = np.array(mask_img)

        # Dataset-specific mask logic
        if "RoadAnomaly" in mask_path:
            gt_bin = np.where(gt_bin == 2, 1, gt_bin)
        elif "LostAndFound" in mask_path:
            gt_bin = np.where(gt_bin == 0, 255, gt_bin)
            gt_bin = np.where(gt_bin == 1, 0, gt_bin)
            gt_bin = np.where((gt_bin > 1) & (gt_bin < 201), 1, gt_bin)
        elif "StreetHazard" in mask_path or "Streethazard" in mask_path:
            gt_bin = np.where(gt_bin == 14, 255, gt_bin)
            gt_bin = np.where(gt_bin < 20, 0, gt_bin)
            gt_bin = np.where(gt_bin == 255, 1, gt_bin)

        is_valid = True
        if 1 not in np.unique(gt_bin):
            is_valid = False

        # Convert mask to tensor
        gt_tensor = torch.from_numpy(gt_bin)
        valid_mask = (gt_tensor == 0) | (gt_tensor == 1)
        labels = gt_tensor[valid_mask]

        return img, valid_mask, labels, is_valid

In [3]:
def compute_anomaly(logits, method, temperature=1.0):
    if method == "msp":
        scaled_logits = logits / temperature
        probs = torch.softmax(scaled_logits, dim=0)
        anomaly = 1 - torch.max(probs, dim=0)[0]

    elif method == "maxlogit":
        anomaly = -torch.max(logits, dim=0)[0]

    elif method == "entropy":
        probs = torch.softmax(logits, dim=0)
        anomaly = -torch.sum(probs * torch.log(probs + 1e-10), dim=0)

    elif method == "rba":
        anomaly = -torch.sum(torch.tanh(logits), dim=0)

    else:
        raise ValueError("Unknown method.")

    return anomaly

In [4]:
def evaluate_anomaly(scores, labels):
    """
    Given a 1D tensor of scores (associated to a method and a temperature), and
    a 1D tensor of labels, it returns the AUPRC and the FP95 scores.
    """

    auprc_fn = BinaryAveragePrecision(thresholds=None).to(device)
    roc_fn = BinaryROC(thresholds=None).to(device)

    auprc = auprc_fn(scores, labels).item()
    fprs, tprs, _ = roc_fn(scores, labels)

    # Find FPR95
    idx = torch.where(tprs >= 0.95)[0]
    fpr95 = fprs[idx[0]].item() if len(idx) > 0 else float('nan')

    auprc_fn.reset()
    roc_fn.reset()

    return auprc, fpr95

In [5]:
def semantic_inference(model, img, device, return_logits=False):
    with torch.no_grad(), torch.autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
          mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
          mask_logits, class_logits_per_layer[-1]
        )

        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)

        if return_logits:
          return logits[0] # [C, H, W]

        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()

    return pred_array

def get_anomaly_scores(model, dataset, tasks, device=0):
    """
    Executes a forward pass of the model on each image of the dataset and
    computes all anomaly scores described in the tasks. Returns a dictionary of
    scores with key (method, temperature) and with value a list of score tensors,
    and also returns a list of label tensors (the anomaly ground truths).
    The tensors in those lists must be concatenated before use.
    """

    # Re-initialize dataloader for this pass
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False,
                            num_workers=2, pin_memory=True)

    # Dictionaries to store the scores for only the current chunk
    scores_dict = { (task[0], task[1]): [] for task in tasks }
    labels_list = []

    with torch.inference_mode():
        for img, valid_mask, labels, is_valid in tqdm(dataloader, desc="Inference pass"):
            if not is_valid.item():
                continue

            img = img.squeeze(0).to(device)
            valid_mask = valid_mask.squeeze(0).to(device)

            labels = labels.squeeze(0).to(device, dtype=torch.int8)

            logits = semantic_inference(model, img, device, return_logits=True)

            for method_type, temp, _ in tasks:
                anomaly = compute_anomaly(logits, method_type, temp)#.half()

                # Extract only valid pixels
                scores = anomaly[valid_mask].float()
                scores_dict[(method_type, temp)].append(scores)
                del anomaly, scores

            # Append the masked labels once per image
            labels_list.append(labels)

            del logits, img, valid_mask, labels

    torch.cuda.empty_cache()

    return scores_dict, labels_list

# Anomaly scores evaluation

In [6]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" # avoid segmentation to get more often free CUDA memory

results = []

models_to_evaluate = {
    "EoMT_Cityscapes": {
        "config": cs_config_path,
        "ckpt": cs_ckpt_path
    },
    "EoMT_COCO": {
        "config": coco_config_path,
        "ckpt": coco_ckpt_path
    },
    "EoMT_COCO_Finetuned": {
        "config": coco_ft_config_path,
        "ckpt": coco_ft_ckpt_path
    },
}

# For MSP, we perform a grid search for all temperatures between 0.1 and 2
# with 0.1 increments, and also include the mandatory temperature 0.75
grid_search_temps = list(np.round(np.arange(0.1, 2.1, 0.1), 2))
msp_temps = sorted(list([0.75] + grid_search_temps))

configs = [
    {"method": "msp",  "name": "MSP", "temps": msp_temps},
    {"method": "maxlogit","name": "MaxLogit", "temps": [1.0]},
    {"method": "entropy", "name": "MaxEntropy", "temps": [1.0]},
    {"method": "rba", "name": "RbA", "temps": [1.0]}
]

# Flatten configs into a list of individual tasks: (method, temp, display_name)
all_tasks = []
for config in configs:
    for temp in config["temps"]:
        all_tasks.append((config["method"], temp, config["name"]))

# Chunk the tasks to prevent OOM. 8 because 24/8 = 3 chunks.
CHUNK_SIZE = 8
task_chunks = [all_tasks[i:i + CHUNK_SIZE] for i in range(0, len(all_tasks), CHUNK_SIZE)]

for model_name, model_config in models_to_evaluate.items(): # for each model
    print(f"\nModel: {model_name}")
    model, _ = load_model_and_data(model_config["config"], model_config["ckpt"], data_path, device)

    for subset in subsets: # for each subset
        print(f"  Dataset: {subset}")
        dataset = AnomalyDataset(data_path_anomaly, subset)

        for chunk_idx, task_chunk in enumerate(task_chunks): # for each task chunk

            # --- Calculation of scores ---
            scores_dict, labels_list = get_anomaly_scores(model, dataset, task_chunk, device)

            # --- Evaluation ---

            # Concatenate labels for this chunk
            all_labels = torch.cat(labels_list)
            del labels_list

            for method_type, temp, display_name in task_chunk: # for each task in chunk
                print(f"[{model_name} - {subset}] METHOD {display_name} - t = {temp}")

                # Retrieve and concatenate scores for this chunk
                method_scores_list = scores_dict.pop((method_type, temp))
                method_scores = torch.cat(method_scores_list)
                del method_scores_list

                auprc, fpr95 = evaluate_anomaly(method_scores, all_labels)

                print(f"AUPRC: {auprc:.4f}")
                print(f"FPR95: {fpr95:.4f}\n")

                results.append({
                    "model": model_name,
                    "dataset": subset,
                    "method": display_name,
                    "temperature": temp,
                    "AUPRC": auprc,
                    "FPR95": fpr95
                })

                del method_scores
                torch.cuda.empty_cache()

            del all_labels
            gc.collect()

    del model
    torch.cuda.empty_cache()

df = pd.DataFrame(results)


Model: EoMT_Cityscapes


c:\Users\user\anaconda3\envs\pytorch_gpu\Lib\site-packages\lightning\pytorch\utilities\parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.
c:\Users\user\anaconda3\envs\pytorch_gpu\Lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\cb\pytorch_1000000000000\work\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


Loading weights from ./checkpoints/eomt_cityscapes.bin...
  Dataset: FS_LostFound_full


Inference pass:   0%|          | 0/100 [00:05<?, ?it/s]


RuntimeError: DataLoader worker (pid(s) 14380, 34472) exited unexpectedly